# Play.py Script (Breakout Game)

Defined the required libraries, set the saved DQN model path, specified the Atari environment, and prepared the evaluation settings needed to load the trained agent consistently.

In [1]:
!pip install -U gymnasium ale-py "stable-baselines3[extra]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 10.2 MB/s eta 0:00:00


In [2]:
import gymnasium as gym
import os
import ale_py

gym.register_envs(ale_py)

print("ALE environments registered.")

ALE environments registered.


In [3]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
MODEL_PATH = "/content/drive/MyDrive/final_model"
ENV_ID = "ALE/Breakout-v5"
N_EPISODES = 5
SEED = 42

In [6]:
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(MODEL_PATH + ".zip"):
    raise FileNotFoundError(
        f"Model not found: {MODEL_PATH}.zip\n"
        "Check the exact Google Drive folder and filename."
    )

model = DQN.load(MODEL_PATH)
print(f"Loaded model from {MODEL_PATH}.zip")

Mounted at /content/drive
Loaded model from /content/drive/MyDrive/final_model.zip


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Created the Atari evaluation environment with human rendering and stacked four frames so the observation format matched the CNN-based DQN model used during training.

In [7]:
env = make_atari_env(
    ENV_ID,
    n_envs=1,
    seed=SEED,
    env_kwargs={"render_mode": "human"}
)

env = VecFrameStack(env, n_stack=4)

Loaded the trained DQN model into the prepared environment so the saved policy could be evaluated using the same observation structure from training.

In [8]:
model = DQN.load(MODEL_PATH, env=env)
print("Model loaded successfully.")

Wrapping the env in a VecTransposeImage.
Model loaded successfully.


Ran several evaluation episodes using deterministic action selection, which serves as greedy policy execution in Stable-Baselines3, while displaying the game and printing total rewards.

In [9]:
for episode in range(N_EPISODES):
    obs = env.reset()
    done = [False]
    total_reward = 0

    while not done[0]:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, done, info = env.step(action)
        total_reward += rewards[0]

    print(f"Episode {episode + 1}: Total Reward = {total_reward}")

env.close()

Episode 1: Total Reward = 7.0
Episode 2: Total Reward = 2.0
Episode 3: Total Reward = 0.0
Episode 4: Total Reward = 5.0
Episode 5: Total Reward = 4.0


# Hyperparameter Experiment

**Batch Size and Buffer Size**

**Setup Inports**

In [10]:
import os
import ale_py
import gymnasium as gym
gym.register_envs(ale_py)

import pandas as pd

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy

Defined the shared training constants so every hyperparameter experiment could be executed consistently under the same conditions.

In [11]:
ENV_ID = "ALE/Breakout-v5"
POLICY = "CnnPolicy"
LEARNING_RATE = 0.0005
GAMMA = 0.99
EXPLORATION_FRAC = 0.1
EXPLORATION_INIT = 1.0
EXPLORATION_FINAL = 0.01
TOTAL_STEPS = 30_000
SEED = 42

SAVE_ROOT = "/content/drive/MyDrive/atari-dqn/member2_experiments"
os.makedirs(SAVE_ROOT, exist_ok=True)

Created a reusable experiment function that trains one DQN configuration, evaluates its average reward, saves the model, and returns structured results for later comparison and documentation.

In [12]:
def run_experiment(exp_name, batch_size, buffer_size):
    exp_path = os.path.join(SAVE_ROOT, exp_name)
    os.makedirs(exp_path, exist_ok=True)

    train_env = make_atari_env(ENV_ID, n_envs=1, seed=SEED)
    train_env = VecFrameStack(train_env, n_stack=4)

    eval_env = make_atari_env(ENV_ID, n_envs=1, seed=SEED + 1)
    eval_env = VecFrameStack(eval_env, n_stack=4)

    model = DQN(
        policy=POLICY,
        env=train_env,
        learning_rate=LEARNING_RATE,
        buffer_size=buffer_size,
        learning_starts=10_000,
        batch_size=batch_size,
        gamma=GAMMA,
        train_freq=4,
        target_update_interval=1000,
        exploration_fraction=EXPLORATION_FRAC,
        exploration_initial_eps=EXPLORATION_INIT,
        exploration_final_eps=EXPLORATION_FINAL,
        verbose=0,
        tensorboard_log=f"{exp_path}/tb_logs/"
    )

    print(f"Running {exp_name} | batch={batch_size} | buffer={buffer_size}")
    model.learn(total_timesteps=TOTAL_STEPS, progress_bar=True)
    model.save(f"{exp_path}/{exp_name}_model")

    mean_reward, std_reward = evaluate_policy(
        model,
        eval_env,
        n_eval_episodes=3,
        deterministic=True
    )

    train_env.close()
    eval_env.close()

    return {
        "experiment": exp_name,
        "batch_size": batch_size,
        "buffer_size": buffer_size,
        "learning_rate": LEARNING_RATE,
        "gamma": GAMMA,
        "epsilon_start": EXPLORATION_INIT,
        "epsilon_end": EXPLORATION_FINAL,
        "epsilon_decay_fraction": EXPLORATION_FRAC,
        "mean_reward": mean_reward,
        "std_reward": std_reward
    }

Initialized an empty results container to store each experiment outcome, making it easier to combine all ten runs into one comparison table afterward.

In [13]:
results = []

**Experiment 1 - Batch = 16**

In [14]:
exp1 = run_experiment(
    exp_name="member2_exp1_batch16",
    batch_size=16,
    buffer_size=100_000
)
results.append(exp1)
pd.DataFrame([exp1])

Output()

Running member2_exp1_batch16 | batch=16 | buffer=100000


/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp1_batch16,16,100000,0.0005,0.99,1.0,0.01,0.1,12.666667,2.867442


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 2 -cBatch = 32**

In [15]:
exp2 = run_experiment(
    exp_name="member2_exp2_batch32",
    batch_size=32,
    buffer_size=100_000
)
results.append(exp2)
pd.DataFrame([exp2])

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Running member2_exp2_batch32 | batch=32 | buffer=100000


,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp2_batch32,32,100000,0.0005,0.99,1.0,0.01,0.1,13.333333,1.247219


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 3 - batch = 64**

In [16]:
exp3 = run_experiment(
    exp_name="member2_exp3_batch64",
    batch_size=64,
    buffer_size=100_000
)
results.append(exp3)
pd.DataFrame([exp3])

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Running member2_exp3_batch64 | batch=64 | buffer=100000


,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp3_batch64,64,100000,0.0005,0.99,1.0,0.01,0.1,11.333333,4.988877


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 4 - Batch = 128**

In [17]:
exp4 = run_experiment(
    exp_name="member2_exp4_batch128",
    batch_size=128,
    buffer_size=100_000
)
results.append(exp4)
pd.DataFrame([exp4])

Output()

Running member2_exp4_batch128 | batch=128 | buffer=100000


/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp4_batch128,128,100000,0.0005,0.99,1.0,0.01,0.1,9.666667,4.496913


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 5 - Batch = 32, Buffer = 10,000**

In [18]:
exp5 = run_experiment(
    exp_name="member2_exp5_batch32_buffer10k",
    batch_size=32,
    buffer_size=10_000
)
results.append(exp5)
pd.DataFrame([exp5])

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Running member2_exp5_batch32_buffer10k | batch=32 | buffer=10000


,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp5_batch32_buffer10k,32,10000,0.0005,0.99,1.0,0.01,0.1,11.333333,1.699673


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiments 6 - Batch = 32, Buffer = 25,000**

In [19]:
exp6 = run_experiment(
    exp_name="member2_exp6_batch32_buffer25k",
    batch_size=32,
    buffer_size=25_000
)
results.append(exp6)
pd.DataFrame([exp6])

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Running member2_exp6_batch32_buffer25k | batch=32 | buffer=25000


,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp6_batch32_buffer25k,32,25000,0.0005,0.99,1.0,0.01,0.1,9.0,3.741657


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 7 - Batch = 32, Buffer = 75, 000**

In [20]:
exp7 = run_experiment(
    exp_name="member2_exp7_batch32_buffer75k",
    batch_size=32,
    buffer_size=75_000
)
results.append(exp7)
pd.DataFrame([exp7])

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 4.23GB > 3.80GB
  warnings.warn(


Running member2_exp7_batch32_buffer75k | batch=32 | buffer=75000


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp7_batch32_buffer75k,32,75000,0.0005,0.99,1.0,0.01,0.1,7.666667,1.885618


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 8 - Batch = 32, Buffer = 100,000**

In [21]:
exp8 = run_experiment(
    exp_name="member2_exp8_batch32_buffer100k",
    batch_size=32,
    buffer_size=100_000
)
results.append(exp8)
pd.DataFrame([exp8])

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Running member2_exp8_batch32_buffer100k | batch=32 | buffer=100000


,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp8_batch32_buffer100k,32,100000,0.0005,0.99,1.0,0.01,0.1,12.666667,4.642796


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 9 - Batch = 64, Buffer = 100,000**

In [22]:
exp9 = run_experiment(
    exp_name="member2_exp9_batch64_buffer100k",
    batch_size=64,
    buffer_size=100_000
)
results.append(exp9)
pd.DataFrame([exp9])

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 5.65GB > 4.06GB
  warnings.warn(


Running member2_exp9_batch64_buffer100k | batch=64 | buffer=100000


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp9_batch64_buffer100k,64,100000,0.0005,0.99,1.0,0.01,0.1,16.0,4.546061


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Experiment 10 - Batch = 128, Buffer = 100,000**

In [23]:
exp10 = run_experiment(
    exp_name="member2_exp10_batch128_buffer100k",
    batch_size=128,
    buffer_size=100_000
)
results.append(exp10)
pd.DataFrame([exp10])

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 5.65GB > 2.38GB
  warnings.warn(


Running member2_exp10_batch128_buffer100k | batch=128 | buffer=100000


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward
0,member2_exp10_batch128_buffer100k,128,100000,0.0005,0.99,1.0,0.01,0.1,10.333333,1.699673


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Final Combined Results Table**

Combined all ten Member Two experiments into one table and added behavior notes, making the final submission easier to compare, interpret, and document clearly.

In [24]:
results_df = pd.DataFrame(results)

def noted_behavior(row):
    if row["mean_reward"] >= 20:
        return "Strong reward performance with stable learning behavior."
    elif row["mean_reward"] >= 10:
        return "Moderate performance with acceptable training stability."
    else:
        return "Lower performance; may require longer training or better tuning."

results_df["noted_behavior"] = results_df.apply(noted_behavior, axis=1)
results_df

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,experiment,batch_size,buffer_size,learning_rate,gamma,epsilon_start,epsilon_end,epsilon_decay_fraction,mean_reward,std_reward,noted_behavior
0,member2_exp1_batch16,16,100000,0.0005,0.99,1.0,0.01,0.1,12.666667,2.867442,Moderate performance with acceptable training ...
1,member2_exp2_batch32,32,100000,0.0005,0.99,1.0,0.01,0.1,13.333333,1.247219,Moderate performance with acceptable training ...
2,member2_exp3_batch64,64,100000,0.0005,0.99,1.0,0.01,0.1,11.333333,4.988877,Moderate performance with acceptable training ...
3,member2_exp4_batch128,128,100000,0.0005,0.99,1.0,0.01,0.1,9.666667,4.496913,Lower performance; may require longer training...
4,member2_exp5_batch32_buffer10k,32,10000,0.0005,0.99,1.0,0.01,0.1,11.333333,1.699673,Moderate performance with acceptable training ...
5,member2_exp6_batch32_buffer25k,32,25000,0.0005,0.99,1.0,0.01,0.1,9.000000,3.741657,Lower performance; may require longer training...
6,member2_exp7_batch32_buffer75k,32,75000,0.0005,0.99,1.0,0.01,0.1,7.666667,1.885618,Lower performance; may require longer training...
7,member2_exp8_batch32_buffer100k,32,100000,0.0005,0.99,1.0,0.01,0.1,12.666667,4.642796,Moderate performance with acceptable training ...
8,member2_exp9_batch64_buffer100k,64,100000,0.0005,0.99,1.0,0.01,0.1,16.000000,4.546061,Moderate performance with acceptable training ...
9,member2_exp10_batch128_buffer100k,128,100000,0.0005,0.99,1.0,0.01,0.1,10.333333,1.699673,Moderate performance with acceptable training ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
results_df.to_csv(f"{SAVE_ROOT}/member2_hyperparameter_results.csv", index=False)
print("Saved results to CSV successfully.")

Saved results to CSV successfully.
